In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

In [ ]:
import pandas as pd
from typing import List, Any
from spacy.tokens import Span
from ExtendedDoc import *
from FindEntityInstances import *
from ResolveEntityInstances import *

In [ ]:
df = pd.read_excel("./benchmarks/Benchmark-03-09.xlsx")
df = df[(df['Score'] == 0) | (df['Score'] == 1) | (df['Score'] == 3)]
df.reset_index(inplace=True)
texts = df.Abstract.tolist()
number_texts = len(texts)

In [ ]:
# True Papers
df_T = df[df['Score'] == 3]
texts_T = df_T.Abstract.tolist()
number_texts_T = len(texts_T)

In [ ]:
# False Papers
df_F = df[(df['Score'] < 3) & (df['Score'] >= 0)]
texts_F = df_F.Abstract.tolist()
number_texts_F = len(texts_F)

In [ ]:
def satisfies_ent_req(doc: ExtendedDoc, ent_instances: List[Span], ents: List[Entity], verbose=False) -> bool:
    if verbose:
        print('Ents')
        for ent in ents:
            print(ent)
        print()
        print()

    # Calculate Counts of Groups
    counts = {}
    for ent in ent_instances:
        ent_text = doc.text_lower[ent.start_char:ent.end_char]
        counts[ent_text] = counts.get(ent_text, 0) + 1
    
    if verbose:
        print('Counts')
        for k, v in counts.items():
            print(v, k)
        print()
        print()
    
    if verbose:
        print('Group\'s Lowers')
        for ent in ents:
            print(ent['lowers'])
        print()
        print()

    # Calculate Counts for Lowers
    lowers: Any = {}
    for ent in ents:
        for lower in ent['lowers']:
            lowers[lower] = lowers.get(lower, 0) +  1

    if verbose:
        print('Lowers')
        for k, v in lowers.items():
            print(v, k)
        print()
        print()

    # Calculate Counts for Entities
    for ent in ents:
        ent['count'] = sum([counts[lower] for lower in ent['lowers']]) # type: ignore
        if verbose:
            print(ent['lowers'], ent['count']) # type: ignore
    if verbose:
        print()
        print()

    non_role_labels = {'v', 's', 'm'}
    non_role_ents = list(ent for ent in ents if non_role_labels & ent['labels'])

    cond_length = len(non_role_ents) >= 2 # Must have >= 2 groups
    cond_counts = len([group for group in non_role_ents if group['count'] >= 2]) >= 2 #type: ignore # Must have at least 2 groups with >= 2 mentions

    flag = cond_length and cond_counts

    if verbose:
        print('Entity Flag:', flag)
        print()
        print()
    
    return flag

In [ ]:
# type: ignore
import cProfile, pstats, io
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Clear Cache
import gc
import functools

# gc.collect()
wrappers = [a for a in gc.get_objects() if isinstance(a, functools._lru_cache_wrapper)]
for wrapper in wrappers:
    wrapper.cache_clear()

txts = [clean_text(text) for i, text in enumerate(texts_T) if i in [25]]

y_pred = []
y_true = []
bad = []

verbose = True

i = 0
for doc in tqdm(nlp.pipe(txts, batch_size=1), total=len(txts)):
    if verbose:
        print()
        print('Text', i)
        print(doc.text)
        print()
        print()
    
    
    doc_extended = ExtendedDoc(doc)

    # Find Entity Instances
    find = FindEntityInstances(doc_extended)
    ent_instances = find()

    # Resolve Entity Instances
    resolve = ResolveEntityInstances(doc_extended)
    ents = resolve(ent_instances)

    # Flag
    flag_ent = satisfies_ent_req(doc_extended, ent_instances, ents, verbose=verbose)
    flag_int = False

    # Put Flags Together
    y_pred.append(flag_ent and flag_int)
    y_true.append(True)

    if y_true[-1] != y_pred[-1]:
        bad.append(i)

    i += 1

ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
plt.show()
print(bad)

In [ ]:
filename = "4-ScreenByEntities"


def save(*, mask, counts, errors, suffix):
    outputs = {
        "counts": counts,
        "mask": mask,
        "errors": errors
    }
    
    with open(rf'{filename}-{suffix or 0}.pickle', 'wb') as file:
        pickle.dump(outputs, file)


def load(suffix):
    outputs = {
        "errors": {},
        "counts": {
            None: 0, 
            True: 0, 
            False: 0
        },
        "mask": []
    }

    fn = f'{filename}-{suffix or 0}.pickle'

    # Nothing To Load
    if not os.path.isfile(fn):
        return outputs

    with open(fn, 'rb') as file:
        outputs = pickle.load(file)
    
    return outputs

In [ ]:
papers = pd.read_csv('3_Papers_Screened.csv')
texts = papers.Abstract.tolist()
number_texts = len(texts)

In [ ]:
# type: ignore
suffix = None

outputs = load(suffix)
mask = outputs['mask']
counts = outputs['counts']
errors = outputs['errors']

i = len(mask)

progress = tqdm(nlp.pipe(texts, batch_size=128), total=number_texts)
for doc in progress:
    progress.set_postfix({
        'Errors': counts[None], 
        'Included': counts[True], 
        'Excluded': counts[False]
    }, refresh=True)

    # Auto-Save
    if (i + 1) % 256:
        save(
            mask=mask, 
            counts=counts, 
            errors=errors, 
            suffix=suffix
        )
    
    try:
        flag = False
    except Exception as e:
        flag = None
        errors[i] = e

    counts[flag] += 1
    mask.append(flag)
    
    i += 1